> **Quick start — click *Run All* — no setup needed.**
> Cached PNG figures are displayed by default (`RERUN = False`).
> Set `RERUN = True` and re-run to regenerate figures from the source scripts.


In [ ]:
import subprocess, sys, pathlib, importlib
from IPython.display import Image, display

# ── locate repository root (search upward for lunar/__init__.py) ──────────
_here = pathlib.Path.cwd()
REPO = None
for _p in [_here, *_here.parents]:
    if (_p / "lunar" / "__init__.py").exists():
        REPO = _p
        break
if REPO is None:
    raise RuntimeError("Cannot find REPO root — run from inside Lunar-V2/")

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

FIGS    = REPO / "output" / "figures"
SCRIPTS = REPO / "scripts" / "phase2"


def show_figure(name: str, caption: str = "") -> None:
    """Display a cached PNG from output/figures/."""
    path = FIGS / name
    if not path.exists():
        print(f"[warn] figure not found: {path}")
        return
    display(Image(str(path), width=820))
    if caption:
        from IPython.display import Markdown
        display(Markdown(f"*{caption}*"))


def run_script(rel: str) -> None:
    """Run a Phase-2 script via subprocess, streaming stdout."""
    script = SCRIPTS / rel
    print(f"▶  running {script.relative_to(REPO)} …")
    result = subprocess.run(
        [sys.executable, str(script)],
        cwd=str(REPO),
        capture_output=False,
    )
    if result.returncode != 0:
        print(f"[error] script exited with code {result.returncode}")
    else:
        print("[done]")

print(f"REPO  = {REPO}")
print(f"FIGS  = {FIGS}  (exists: {FIGS.exists()})")
print(f"SCRIPTS = {SCRIPTS}  (exists: {SCRIPTS.exists()})")


# Phase 2 (Craters) — Bowl-crater illumination model

Mirrors the upstream `Craters/` directory from Martinez & Siegler (2021).

Covers two components:

1. **`crater_floor_insolation()`** — Python port of the upstream
   `insolationcrater.m` script.  Computes absorbed solar flux on the floor
   of a generic bowl crater as a function of time.
2. **Horizon tracer** (`lunar/illumination.py`) — Mazarico (2011) method
   for computing sky-fraction / illumination fraction from a DEM.  This is
   the tool that would produce `shoemakerIllumination.mat`-equivalents for
   new sites (see `03_psr_shoemaker.ipynb` for the Shoemaker PSR case).


## §1 — `crater_floor_insolation()` — port of upstream `insolationcrater.m`

The floor-averaged absorbed flux for a bowl crater is:

$$Q_s = \frac{S \cdot 4\varepsilon(1-A)}{D^2} \cdot \left(1 + \frac{A}{\varepsilon}\right) \cdot \max(0,\,\cos\theta)$$

where *S* is the solar constant (1361 W m⁻²), *A* is the bolometric albedo,
*ε* is the floor-to-diameter aspect ratio, *D* is the normalised diameter,
and θ is the solar incidence angle at the crater-floor centre.

> **Note** — this formula is valid for *generic* bowl craters.  For
> **Shoemaker PSR** specifically, the upstream pipeline uses a precomputed
> ray-traced time series (`shoemakerIllumination.mat`); see
> `03_psr_shoemaker.ipynb`.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from lunar.illumination import crater_floor_insolation

# Bowl crater at 70 °S, diameter_norm = D/depth = 3.0 (generic ratio)
t = np.linspace(0, 2.55024e6, 2000)          # one lunation in seconds
Q = crater_floor_insolation(t, latitude_deg=-70.0, diameter_norm=3.0)

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(t / 86400, Q, lw=1.4, color="#1f77b4")
ax.set_xlabel("Time (days)")
ax.set_ylabel("Absorbed flux (W m⁻²)")
ax.set_title("Bowl-crater floor insolation: lat = −70 °, D_norm = 3.0")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## §2 — Horizon tracer (`lunar/illumination.py`)

The horizon tracer implements the **Mazarico (2011)** method:

- For each surface point, scan 360 azimuths and determine the maximum
  elevation angle to the horizon (limited by the DEM grid).
- Illumination fraction = fraction of valid Sun positions above the horizon.

**Key functions**

| Function | Purpose |
|---|---|
| `compute_horizon(dem, ...)` | Compute horizon elevation angles from a `DEM` object |
| `synthetic_crater_dem(...)` | Generate a synthetic bowl-crater DEM for testing |

The `compute_horizon` output is what would produce a
`shoemakerIllumination.mat`-equivalent for any new site.

> For a full horizon-tracer demonstration notebook see
> `notebooks/phase2_illumination/04_illumination_shadows.ipynb` (legacy dir).


In [ ]:
import inspect
from lunar.illumination import compute_horizon, synthetic_crater_dem

print("compute_horizon signature:")
print(inspect.signature(compute_horizon))
print()

doc = compute_horizon.__doc__
if doc:
    print(doc[:400])
else:
    print("(no docstring)")
